### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="blood_transfusion",
    dataset_year="2008",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5GS39",
    download_description="""
We download the data from the UCI repository and unzip it to a predefined folder.

mkdir -p local-data-warehouse/blood_transfusion/ && wget -P local-data-warehouse/blood_transfusion/ https://archive.ics.uci.edu/static/public/176/blood+transfusion+service+center.zip && unzip local-data-warehouse/blood_transfusion/blood+transfusion+service+center.zip -d local-data-warehouse/blood_transfusion/
""",
    # References
    academic_reference_bibtex="""@article{yeh2009knowledge,
  title={Knowledge discovery on RFM model using Bernoulli sequence},
  author={Yeh, I-Cheng and Yang, King-Jang and Ting, Tao-Ming},
  journal={Expert Systems with applications},
  volume={36},
  number={3},
  pages={5866--5871},
  year={2009},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="yeh2009knowledge",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We made feature names more descriptive.
- We renamed the target and mapped binary values to "Yes"/"No".
- Anomaly: the data has a lot of duplicates (29%) and several duplicates with different target values.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="DonatedBloodInMarch2007",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="DonatedBloodInMarch2007",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/transfusion.data")

target_feature = "DonatedBloodInMarch2007"
df.columns = [
    "MonthsSinceLastDonation",
    "NumberOfDonations",
    "TotalBloodDonated",
    "MonthsSinceFirstDonation",
    target_feature,
]
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df[target_feature] = df[target_feature].map({1: "Yes", 0: "No"})

cat_features = [
    "DonatedBloodInMarch2007",
]

df[cat_features] = df[cat_features].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

/tmp/ipykernel_579758/3720450945.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 748
Columns: 5
Use sampling: False (sample size: 748)
Get row duplicates (staged, merged)...
Using top-4 columns for initial filtering: ['MonthsSinceFirstDonation', 'TotalBloodDonated', 'NumberOfDonations', 'MonthsSinceLastDonation']
Rows remaining as candidates after top-4 filter: 315 (of 748)

#### Duplicate Report
Total duplicate rows: 215 (28.74% of dataset)
Duplicate rows ignoring target: 246 (32.89% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,MonthsSinceLastDonation,NumberOfDonations,TotalBloodDonated,MonthsSinceFirstDonation,DonatedBloodInMarch2007
0,2,1,250,2,No
1,16,6,1500,40,No
2,4,6,1500,35,No
3,11,2,500,11,No
4,14,2,500,14,No


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,DonatedBloodInMarch2007,category,0.0,0.0,2.0,"No, Yes"
1,MonthsSinceLastDonation,int64,0.0,0.0,31.0,"2, 4, 11, 14, 16, 23, 21, 9, 3, 1"
2,NumberOfDonations,int64,0.0,0.0,33.0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 11"
3,TotalBloodDonated,int64,0.0,0.0,33.0,"250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2750"
4,MonthsSinceFirstDonation,int64,0.0,0.0,78.0,"4, 16, 14, 23, 2, 28, 26, 11, 35, 21"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MonthsSinceLastDonation,748.0,9.506684,8.095396,0.0,74.0
NumberOfDonations,748.0,5.514706,5.839307,1.0,50.0
TotalBloodDonated,748.0,1378.676471,1459.826781,250.0,12500.0
MonthsSinceFirstDonation,748.0,34.282086,24.376714,2.0,98.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column                  rank                   
DonatedBloodInMarch2007 1       No    570  76.2
                        2      Yes    178  23.8

In [8]:
# Target Distribution
target_df

,count,pct
DonatedBloodInMarch2007,,
No,570,76.2
Yes,178,23.8


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to blood_transfusion/019d7366-d9d0-777d-b0ae-e7a5175f7f09


019d7366-d9d0-777d-b0ae-e7a5175f7f09
3f921281644172b0ae48b7180b852a6d67ef7e867e5ebcfc05c891bf759b6ebb
